# UCI Phishing Websites DDLGN Depth vs Width Sweep

This notebook trains Differentiable Logic Gate Networks (DDLGNs) on the UCI **Phishing Websites** tabular dataset for the same depth/width grid used in the MNIST and Fashion-MNIST experiments.

The dataset has:

- 11,055 website examples.
- 30 integer-valued features extracted from URL, domain, HTML, and web-traffic properties.
- A binary target stored as `Result`, where the original ARFF labels are `-1` and `1`.
- No missing values according to the UCI dataset card.

Source: Mohammad, R. & McCluskey, L. (2012). *Phishing Websites*. UCI Machine Learning Repository. https://doi.org/10.24432/C51W2X

For LGN training, each ternary feature value (`-1`, `0`, `1`) is one-hot encoded into three boolean inputs. That creates a 90-bit input vector (`30 features * 3 values`) while preserving the original categorical/integer feature information.


## Environment to activate

Use the same Windows CUDA environment described by `../../docs/DiffLogic_Installation_Guide.pdf` before starting Jupyter:

```powershell
conda activate difflogic2
cd <path-to-EI-DDLGN>
jupyter lab
```

Expected Python side:

- Python `3.10`
- PyTorch `2.5.1` with CUDA `12.1`
- CUDA Toolkit `12.1`
- `difflogic` built from the patched source described in the installation guide
- `difflogic_cuda` importable from the active environment
- `pandas`, `numpy`, and `tqdm`

Expected Rust side:

- `cargo` available on `PATH`
- the repository's `MNIST_DDLGNs/lgn_eval/Cargo.toml` present

The next cell checks these assumptions before the long training sweep begins.


In [ ]:
# Preflight: project paths, CUDA DLLs, Python packages, and Rust tooling.
import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path

REQUIRE_CUDA = True

def find_project_dir():
    candidates = [
        Path.cwd(),
        *Path.cwd().parents,
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "crates" / "ei-ddlgn-eval" / "Cargo.toml").exists():
            return candidate.resolve()
    raise RuntimeError("Could not locate MNIST_DDLGNs. Start Jupyter from the MNIST_DDLGNs folder or the repo root.")

PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
print("Working directory:", PROJECT_DIR)
print("Python executable:", sys.executable)
print("Platform:", platform.platform())

import torch

if os.name == "nt":
    torch_lib_dir = Path(torch.__file__).resolve().parent / "lib"
    cuda_bin_dir = Path(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin")
    for dll_dir in [torch_lib_dir, cuda_bin_dir]:
        if dll_dir.exists():
            os.add_dll_directory(str(dll_dir))
            print("Added DLL directory:", dll_dir)
        else:
            print("DLL directory not found:", dll_dir)

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
elif REQUIRE_CUDA:
    raise RuntimeError("CUDA is not available. Activate the difflogic2 CUDA environment before running this notebook.")

import torchvision
import pandas as pd
import numpy as np
from tqdm import tqdm

import difflogic
from difflogic import LogicLayer, GroupSum, PackBitsTensor
from difflogic.compiled_model import ALL_OPERATIONS
print("difflogic:", difflogic.__file__)

if REQUIRE_CUDA:
    import difflogic_cuda
    print("difflogic_cuda loaded:", difflogic_cuda)

cargo_path = shutil.which("cargo")
if cargo_path is None:
    raise RuntimeError("cargo was not found on PATH. Install Rust with rustup and restart the terminal/Jupyter server.")
cargo_version = subprocess.run(["cargo", "--version"], capture_output=True, text=True, check=True).stdout.strip()
print("cargo:", cargo_version)

RUST_MANIFEST = PROJECT_DIR / "crates" / "ei-ddlgn-eval" / "Cargo.toml"
if not RUST_MANIFEST.exists():
    raise RuntimeError(f"Missing Rust manifest: {RUST_MANIFEST}")
print("Rust manifest:", RUST_MANIFEST)


## Sweep configuration

The architecture grid mirrors the MNIST depth-vs-width table:

- `depth`: number of `LogicLayer` layers, from `1` to `6`.
- `width`: number of logic gates per layer: `2,000`, `4,000`, `6,000`, and `8,000`.

Because this is a binary classification problem, the final `GroupSum` has two output classes. All widths in this grid are divisible by `2`, so each class receives the same number of grouped logic outputs.


In [ ]:
import datetime
import json
import random
import time
import pickle
import csv
import gc
import urllib.request
import zipfile
from copy import deepcopy

from torch.utils.data import TensorDataset, DataLoader

WIDTHS = [2000, 4000, 6000, 8000]
DEPTHS = [1, 2, 3, 4, 5, 6]
CLASS_COUNT = 2

TAU_BY_WIDTH = {
    2000: 3.0,
    4000: 3.0,
    6000: 5.0,
    8000: 10.0,
}

BASE_ARGS = {
    "id": 327,
    "dataset": "uci_phishing_websites",
    "batch_size": 256,
    "learning_rate": 0.01,
    "epochs": 200,
    "patience": 10,
    "seed": 952,
    "train_fraction": 0.70,
    "val_fraction": 0.15,
    "test_fraction": 0.15,
    "input_encoding": "ternary_one_hot",
    "original_feature_values": [-1, 0, 1],
    "requested_connections": "unique",
    "fallback_connections": "random",
    "connection_policy": "Use unique input pairs when possible; use random connections only when width exceeds the unique-pair capacity of a layer.",
}

SWEEP_ROOT = PROJECT_DIR / "outputs" / "training" / "uci-phishing"
SWEEP_RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
SWEEP_OUTPUT_DIR = SWEEP_ROOT / SWEEP_RUN_ID
SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = PROJECT_DIR / "data" / "uci-phishing"
RAW_DIR = DATA_DIR / "raw"
DATA_URL = "https://archive.ics.uci.edu/static/public/327/phishing+websites.zip"
TARGET_COLUMN = "Result"
ORIGINAL_LABEL_TO_INDEX = {-1: 0, 1: 1}
INDEX_TO_CLASS_NAME = {0: "phishing (-1)", 1: "legitimate (1)"}

RUN_SWEEP = True
SKIP_EXISTING = True

print("Sweep output directory:", SWEEP_OUTPUT_DIR)
print("Architectures:", [(depth, width) for depth in DEPTHS for width in WIDTHS])


## Download and parse the UCI dataset

The UCI download contains an ARFF file named `Training Dataset.arff`. This cell downloads the zip if needed and parses the ARFF directly, so it does not require `scipy` or `ucimlrepo`.

The original 30 feature columns use integer values `-1`, `0`, and `1`. The exact meaning of those values depends on the feature, as described in the UCI feature document. For DDLGN training, the values are treated as categorical states and one-hot encoded into boolean features.


In [ ]:
def download_uci_phishing_dataset():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "phishing_websites.zip"
    if not zip_path.exists():
        print("Downloading:", DATA_URL)
        urllib.request.urlretrieve(DATA_URL, zip_path)
    else:
        print("Using existing download:", zip_path)

    marker = RAW_DIR / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(RAW_DIR)
        marker.write_text(datetime.datetime.now().isoformat(timespec="seconds"), encoding="utf-8")
    return zip_path


def parse_arff_integer_dataset(arff_path):
    attributes = []
    rows = []
    in_data = False

    with open(arff_path, "r", encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("%"):
                continue
            lower = line.lower()
            if lower.startswith("@attribute"):
                parts = line.split(None, 2)
                if len(parts) < 3:
                    raise ValueError(f"Could not parse ARFF attribute line: {line}")
                name = parts[1].strip().strip("'\"")
                attributes.append(name)
            elif lower.startswith("@data"):
                in_data = True
            elif in_data:
                values = [int(value.strip()) for value in line.split(",")]
                rows.append(values)

    if not attributes or not rows:
        raise RuntimeError(f"No attributes or rows parsed from {arff_path}")

    return pd.DataFrame(rows, columns=attributes)


zip_path = download_uci_phishing_dataset()
arff_candidates = sorted(RAW_DIR.rglob("Training Dataset.arff"))
if not arff_candidates:
    raise FileNotFoundError(f"Could not find Training Dataset.arff under {RAW_DIR}")

ARFF_PATH = arff_candidates[0]
raw_df = parse_arff_integer_dataset(ARFF_PATH)
feature_names = [column for column in raw_df.columns if column != TARGET_COLUMN]

print("Downloaded zip:", zip_path)
print("ARFF path:", ARFF_PATH)
print("Raw shape:", raw_df.shape)
print("Feature count:", len(feature_names))
print("Target values:", sorted(raw_df[TARGET_COLUMN].unique().tolist()))
print(raw_df.head().to_string())


## Dataset checks and binary encoding

The DDLGN model consumes boolean inputs. This cell converts each original feature into three indicator columns:

- `<feature>__eq_-1`
- `<feature>__eq_0`
- `<feature>__eq_1`

The target labels are mapped as:

- original `-1` -> index `0` -> `phishing (-1)`
- original `1` -> index `1` -> `legitimate (1)`

This mapping is saved into every model's metadata JSON so the exported reports are reconstructable.


In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def encode_ternary_features_as_binary(df, feature_columns, values=(-1, 0, 1)):
    encoded_columns = []
    encoded_names = []
    for feature in feature_columns:
        column = df[feature].to_numpy()
        for value in values:
            encoded_columns.append((column == value).astype(np.float32))
            encoded_names.append(f"{feature}__eq_{value}")
    X = np.stack(encoded_columns, axis=1).astype(np.float32)
    return X, encoded_names


def encode_labels(series):
    labels = series.astype(int).to_numpy()
    unexpected = sorted(set(labels.tolist()) - set(ORIGINAL_LABEL_TO_INDEX.keys()))
    if unexpected:
        raise ValueError(f"Unexpected label values: {unexpected}")
    return np.array([ORIGINAL_LABEL_TO_INDEX[int(label)] for label in labels], dtype=np.int64)


set_seed(BASE_ARGS["seed"])

if raw_df.isna().any().any():
    raise RuntimeError("The parsed dataset contains missing values; preprocessing must be updated before training.")

for feature in feature_names:
    unexpected = sorted(set(raw_df[feature].astype(int).unique().tolist()) - set(BASE_ARGS["original_feature_values"]))
    if unexpected:
        raise ValueError(f"Feature {feature} has values outside -1/0/1: {unexpected}")

X_binary, binary_feature_names = encode_ternary_features_as_binary(
    raw_df,
    feature_names,
    values=BASE_ARGS["original_feature_values"],
)
y = encode_labels(raw_df[TARGET_COLUMN])
INPUT_DIM = int(X_binary.shape[1])

encoded_df = pd.DataFrame(X_binary.astype(np.int8), columns=binary_feature_names)
encoded_df.insert(0, "label", y)

print("Original feature matrix shape:", raw_df[feature_names].shape)
print("Binary feature matrix shape:", X_binary.shape)
print("Class counts:")
print(pd.Series(y).map(INDEX_TO_CLASS_NAME).value_counts())
print(encoded_df.head().to_string())


## Visualization: class balance

This cell checks whether the phishing and legitimate classes are balanced enough for the depth-vs-width experiment. The split cell below uses stratification so every train/validation/test split keeps approximately the same class proportions.


In [ ]:
class_counts = pd.Series(y).map(INDEX_TO_CLASS_NAME).value_counts().sort_index()
class_percent = class_counts / class_counts.sum() * 100.0

# These visualization cells only write HTML files and print short text summaries.
# Avoiding inline rendering keeps VS Code/Jupyter out of the chart drawing path.
def _html_escape(value):
    return str(value).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def _save_html_chart(filename, title, body_html):
    out_path = SWEEP_OUTPUT_DIR / filename
    html = f"""<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>{_html_escape(title)}</title>
<style>
body {{ font-family: Segoe UI, Arial, sans-serif; margin: 24px; color: #0f172a; }}
h1 {{ font-size: 20px; margin: 0 0 16px 0; }}
</style>
</head>
<body>
<h1>{_html_escape(title)}</h1>
{body_html}
</body>
</html>"""
    out_path.write_text(html, encoding="utf-8")
    print("Saved HTML chart:", out_path)
    return out_path

rows = []
colors = ["#c2410c", "#0f766e"]
max_count = float(class_counts.max())
for idx, (label, count) in enumerate(class_counts.items()):
    pct = float(class_percent.loc[label])
    bar_width = 100.0 * float(count) / max_count
    rows.append(f"""
      <div style="display:grid;grid-template-columns:160px 1fr 120px;align-items:center;gap:10px;margin:8px 0;max-width:820px;">
        <div style="font-size:13px;">{_html_escape(label)}</div>
        <div style="background:#f1f5f9;border-radius:4px;height:26px;position:relative;">
          <div style="background:{colors[idx % len(colors)]};width:{bar_width:.2f}%;height:26px;border-radius:4px;"></div>
        </div>
        <div style="font-size:13px;text-align:right;">{int(count)} ({pct:.1f}%)</div>
      </div>
    """)

chart_html = "\n".join(rows)
_save_html_chart("uci_phishing_class_balance.html", "UCI Phishing Websites class balance", chart_html)

balance_table = pd.DataFrame({"count": class_counts, "percent": class_percent.round(2)})
balance_csv = SWEEP_OUTPUT_DIR / "uci_phishing_class_balance.csv"
balance_table.to_csv(balance_csv)
print("Saved table:", balance_csv)
print(balance_table.to_string())


## Visualization: feature value distribution

Each original feature is a ternary integer column. This stacked chart shows how often each feature takes values `-1`, `0`, and `1`. Features with a high fraction of a single value may carry less information; features with class-separated values are usually more useful.


In [ ]:
value_columns = BASE_ARGS["original_feature_values"]
feature_value_counts = raw_df[feature_names].apply(pd.Series.value_counts).fillna(0).T[value_columns]
feature_value_percent = feature_value_counts.div(feature_value_counts.sum(axis=1), axis=0) * 100.0

feature_value_summary = feature_value_percent.round(2)
summary_csv = SWEEP_OUTPUT_DIR / "uci_phishing_feature_value_distribution.csv"
feature_value_summary.to_csv(summary_csv)
print("Saved table:", summary_csv)
print(feature_value_summary.head(12).to_string())

stack_colors = {-1: "#c2410c", 0: "#64748b", 1: "#0f766e"}
rows = []
for feature, row in feature_value_percent.head(12).iterrows():
    parts = []
    for value_name in value_columns:
        pct = float(row[value_name])
        if pct > 0:
            parts.append(
                f'<div title="{_html_escape(value_name)}: {pct:.1f}%" '
                f'style="background:{stack_colors[value_name]};width:{pct:.2f}%;height:24px;"></div>'
            )
    rows.append(f"""
      <div style="display:grid;grid-template-columns:260px 1fr;align-items:center;gap:10px;margin:7px 0;max-width:920px;">
        <div title="{_html_escape(feature)}" style="font-size:12px;text-align:right;">{_html_escape(feature[:34])}</div>
        <div style="display:flex;background:#f1f5f9;border-radius:4px;overflow:hidden;height:24px;">{''.join(parts)}</div>
      </div>
    """)
legend = """
<div style="display:flex;gap:16px;margin:0 0 12px 270px;font-size:12px;">
  <span><span style="display:inline-block;width:10px;height:10px;background:#c2410c;"></span> -1</span>
  <span><span style="display:inline-block;width:10px;height:10px;background:#64748b;"></span> 0</span>
  <span><span style="display:inline-block;width:10px;height:10px;background:#0f766e;"></span> 1</span>
</div>
"""
_save_html_chart("uci_phishing_feature_value_distribution.html", "Feature value distribution", legend + "\n".join(rows))


## Visualization: feature association with the target

The original target uses `-1` and `1`, so a Pearson correlation on the ternary feature columns is a compact first look at which features are most associated with the class label. This is exploratory only; the DDLGN still receives the full one-hot encoded feature vector.


In [ ]:
import math

# Avoid pandas built-in correlation here. In this Windows CUDA environment, pandas/numpy
# correlation can abort the kernel with a duplicate OpenMP runtime after torch is loaded.
def pearson_corr_plain_python(x_values, y_values):
    x = [float(value) for value in x_values]
    y = [float(value) for value in y_values]
    n = len(x)
    if n != len(y) or n == 0:
        raise ValueError("Correlation inputs must have the same nonzero length.")
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    dx = [value - mean_x for value in x]
    dy = [value - mean_y for value in y]
    denom_x = sum(value * value for value in dx)
    denom_y = sum(value * value for value in dy)
    denom = math.sqrt(denom_x * denom_y)
    if denom == 0:
        return 0.0
    return sum(a * b for a, b in zip(dx, dy)) / denom


def _short_label(label, max_chars=34):
    label = str(label)
    if len(label) <= max_chars:
        return label
    return label[: max_chars - 1] + "..."

target_original = raw_df[TARGET_COLUMN].tolist()
feature_corr_values = {
    feature: pearson_corr_plain_python(raw_df[feature].tolist(), target_original)
    for feature in feature_names
}
sorted_features = sorted(feature_corr_values, key=lambda feature: abs(feature_corr_values[feature]), reverse=True)
top_corr = pd.Series({feature: feature_corr_values[feature] for feature in sorted_features[:15]}, name="corr_with_Result")

corr_csv = SWEEP_OUTPUT_DIR / "uci_phishing_top_feature_correlations.csv"
top_corr.to_frame().to_csv(corr_csv)
print("Saved table:", corr_csv)
print(top_corr.to_frame().to_string())

rows = []
for feature, value in reversed(list(top_corr.items())):
    value = float(value)
    left_width = max(0.0, -value) * 50.0
    right_width = max(0.0, value) * 50.0
    color = "#0f766e" if value >= 0 else "#c2410c"
    rows.append(f"""
      <div style="display:grid;grid-template-columns:260px 1fr 70px;align-items:center;gap:8px;margin:6px 0;max-width:1050px;">
        <div title="{_html_escape(feature)}" style="font-size:12px;text-align:right;">{_html_escape(_short_label(feature))}</div>
        <div style="display:grid;grid-template-columns:1fr 1fr;height:20px;background:linear-gradient(to right, transparent calc(50% - 1px), #111827 calc(50% - 1px), #111827 calc(50% + 1px), transparent calc(50% + 1px));border:1px solid #e2e8f0;">
          <div style="display:flex;justify-content:flex-end;"><div style="background:{color};width:{left_width:.3f}%;height:20px;"></div></div>
          <div><div style="background:{color};width:{right_width:.3f}%;height:20px;"></div></div>
        </div>
        <div style="font-size:12px;text-align:right;">{value:.3f}</div>
      </div>
    """)

axis_html = """
<div style="display:grid;grid-template-columns:260px 1fr 70px;gap:8px;margin:0 0 6px 0;color:#475569;font-size:11px;max-width:1050px;">
  <div></div>
  <div style="display:grid;grid-template-columns:repeat(5,1fr);text-align:center;"><span>-1.0</span><span>-0.5</span><span>0</span><span>0.5</span><span>1.0</span></div>
  <div></div>
</div>
"""
_save_html_chart("uci_phishing_top_feature_correlations.html", "Top feature correlations", axis_html + "\n".join(rows))


## Train/validation/test split

This split is deterministic and stratified. The default fractions are `70%` train, `15%` validation, and `15%` test. The validation set drives early stopping; the test set is exported as `test_binarized.csv` for the Python and Rust plaintext evaluators.


In [ ]:
def stratified_split_indices(labels, train_fraction, val_fraction, seed):
    labels = np.asarray(labels)
    rng = np.random.default_rng(seed)
    train_indices = []
    val_indices = []
    test_indices = []

    for cls in sorted(np.unique(labels).tolist()):
        cls_indices = np.where(labels == cls)[0]
        rng.shuffle(cls_indices)
        n = len(cls_indices)
        n_train = int(round(train_fraction * n))
        n_val = int(round(val_fraction * n))
        n_train = min(n_train, n)
        n_val = min(n_val, n - n_train)
        train_indices.extend(cls_indices[:n_train].tolist())
        val_indices.extend(cls_indices[n_train:n_train + n_val].tolist())
        test_indices.extend(cls_indices[n_train + n_val:].tolist())

    for indices in [train_indices, val_indices, test_indices]:
        rng.shuffle(indices)

    return np.array(train_indices), np.array(val_indices), np.array(test_indices)


train_idx, val_idx, test_idx = stratified_split_indices(
    y,
    BASE_ARGS["train_fraction"],
    BASE_ARGS["val_fraction"],
    BASE_ARGS["seed"],
)

X_train, y_train = X_binary[train_idx], y[train_idx]
X_val, y_val = X_binary[val_idx], y[val_idx]
X_test, y_test = X_binary[test_idx], y[test_idx]

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_dataset = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=BASE_ARGS["batch_size"], shuffle=True, drop_last=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BASE_ARGS["batch_size"], shuffle=False, drop_last=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BASE_ARGS["batch_size"], shuffle=False, drop_last=False, num_workers=0)

split_rows = []
for name, labels in [("train", y_train), ("val", y_val), ("test", y_test)]:
    counts = pd.Series(labels).map(INDEX_TO_CLASS_NAME).value_counts().sort_index()
    split_rows.append({
        "split": name,
        "rows": int(len(labels)),
        **{class_name: int(counts.get(class_name, 0)) for class_name in INDEX_TO_CLASS_NAME.values()},
    })

split_df = pd.DataFrame(split_rows)
print(split_df.to_string())

x0, y0 = next(iter(train_loader))
print("Batch shape:", x0.shape, y0.shape)
print("Input dimension:", INPUT_DIM)


## Training and export helpers

The export format intentionally mirrors the existing MNIST trained model folders:

- `log_args.txt`
- `log_train.txt`
- `log_val.txt`
- `log_seconds.txt`
- `Model_depth-XX_width-YYYYY_epoch-Z`
- `best_lgn_gates.csv`
- `best_lgn_metadata.json`
- `test_binarized.csv`
- `results_boolean_depth-XX_width-YYYYY.txt`
- `training_summary.json`

The base helper functions below are reused from the MNIST depth-vs-width notebook. The following cell overrides the dataset-specific metadata and input-dimension pieces for this tabular dataset.


In [ ]:
def make_args(depth, width):
    args = dict(BASE_ARGS)
    args.update({
        "num_layers": int(depth),
        "num_neurons": int(width),
        "tau": float(TAU_BY_WIDTH[int(width)]),
        "sweep_depth": int(depth),
        "sweep_width": int(width),
    })
    return args

def compute_in_dim(args):
    if args["pixel_mode"] == "multi_threshold":
        return int(args["num_niveis"] * args["img_size"] * args["img_size"])
    if args["pixel_mode"] == "threshold_05":
        return int(args["img_size"] * args["img_size"])
    raise ValueError(f"Unknown pixel_mode: {args['pixel_mode']}")


def unique_pair_capacity(in_dim):
    in_dim = int(in_dim)
    return in_dim * (in_dim - 1) // 2


def choose_logic_connections(layer_in_dim, layer_out_dim, args):
    layer_in_dim = int(layer_in_dim)
    layer_out_dim = int(layer_out_dim)
    requested = args.get("requested_connections", args.get("connections", "unique"))
    fallback = args.get("fallback_connections", "random")

    if requested not in {"unique", "random"}:
        raise ValueError(f"Unsupported requested_connections={requested!r}")
    if fallback not in {"random"}:
        raise ValueError(f"Unsupported fallback_connections={fallback!r}; only 'random' is supported here.")

    if layer_out_dim * 2 < layer_in_dim:
        raise ValueError(
            f"LogicLayer requires out_dim * 2 >= in_dim, but got in_dim={layer_in_dim}, out_dim={layer_out_dim}."
        )

    capacity = unique_pair_capacity(layer_in_dim)
    used_fallback = False
    reason = None
    connections = requested
    if requested == "unique" and layer_out_dim > capacity:
        connections = fallback
        used_fallback = True
        reason = (
            f"unique pair capacity for in_dim={layer_in_dim} is {capacity}, "
            f"which is smaller than out_dim={layer_out_dim}"
        )

    return connections, {
        "requested_connections": requested,
        "connections": connections,
        "fallback_connections": fallback,
        "used_fallback": used_fallback,
        "fallback_reason": reason,
        "unique_pair_capacity": int(capacity),
    }


def build_lgn(args, device):
    in_dim = compute_in_dim(args)
    width = int(args["num_neurons"])
    depth = int(args["num_layers"])
    llkw = dict(grad_factor=1.0)

    logic_layers = [torch.nn.Flatten()]
    connection_plan = []

    def append_logic_layer(layer_in_dim, layer_out_dim):
        model_layer_index = len(logic_layers)
        logic_layer_index = len(connection_plan)
        connections, plan = choose_logic_connections(layer_in_dim, layer_out_dim, args)
        plan.update({
            "logic_layer_index": int(logic_layer_index),
            "model_layer_index": int(model_layer_index),
            "in_dim": int(layer_in_dim),
            "out_dim": int(layer_out_dim),
        })
        if plan["used_fallback"]:
            print(
                f"Connection fallback for logic layer {logic_layer_index}: "
                f"requested unique but using random because {plan['fallback_reason']}."
            )
        connection_plan.append(plan)
        return LogicLayer(in_dim=layer_in_dim, out_dim=layer_out_dim, device=device, connections=connections, **llkw)

    logic_layers.append(append_logic_layer(in_dim, width))
    for _ in range(depth - 1):
        logic_layers.append(append_logic_layer(width, width))

    model = torch.nn.Sequential(*logic_layers, GroupSum(CLASS_COUNT, args["tau"]))
    model = model.to(device)
    model.implementation = "cuda" if device == "cuda" else "python"
    model.connection_plan = connection_plan

    total_num_neurons = sum(layer.num_neurons for layer in logic_layers[1:])
    total_num_weights = sum(layer.num_weights for layer in logic_layers[1:])
    return model, in_dim, total_num_neurons, total_num_weights

def train_step(model, x, y, loss_fn, optimizer):
    out = model(x)
    loss = loss_fn(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.item())

def _get_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")

def eval_model(model, loader, mode=False, device=None):
    device = torch.device(device) if device is not None else _get_model_device(model)
    orig_mode = model.training
    with torch.no_grad():
        model.train(mode=mode)
        batch_accs = []
        for x, y in loader:
            x = x.to(torch.float32).to(device).round()
            y = y.to(device)
            batch_accs.append((model(x).argmax(-1) == y).to(torch.float32).mean().item())
        model.train(mode=orig_mode)
    return float(np.mean(batch_accs))

def export_lgn_gates_csv_and_json(model, csv_path, json_path, args, class_count, extra_metadata=None):
    rows = []
    for layer_idx, layer in enumerate(model):
        if isinstance(layer, LogicLayer):
            gate_ids = layer.weights.argmax(1).detach().cpu().tolist()
            a_idx = layer.indices[0].detach().cpu().tolist()
            b_idx = layer.indices[1].detach().cpu().tolist()
            for neuron_idx, (a, b, gate_id) in enumerate(zip(a_idx, b_idx, gate_ids)):
                rows.append({
                    "layer": int(layer_idx),
                    "neuron": int(neuron_idx),
                    "input_a": int(a),
                    "input_b": int(b),
                    "gate_id": int(gate_id),
                    "gate_name": str(ALL_OPERATIONS[gate_id]),
                })

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        )
        writer.writeheader()
        writer.writerows(rows)

    pixel_mode = args.get("pixel_mode", "threshold_05")
    num_niveis = int(args.get("num_niveis", 1))
    img_size = int(args["img_size"])
    if pixel_mode == "multi_threshold":
        in_dim = num_niveis * img_size * img_size
        binarization = {
            "type": "levels_thresholds",
            "num_niveis": num_niveis,
            "thresholds": [(i + 1) / float(num_niveis + 1) for i in range(num_niveis)],
            "comparison": ">",
        }
    elif pixel_mode == "threshold_05":
        in_dim = img_size * img_size
        binarization = {"type": "single_threshold", "threshold": 0.5, "comparison": ">"}
    else:
        raise ValueError(f"Unknown pixel_mode: {pixel_mode}")

    logic_layer_sizes = [int(layer.out_dim) for layer in model if isinstance(layer, LogicLayer)]
    meta = {
        "class_count": int(class_count),
        "img_size": img_size,
        "pixel_mode": pixel_mode,
        "num_niveis": num_niveis,
        "in_dim": int(in_dim),
        "num_layers": int(args["num_layers"]),
        "num_neurons": int(args["num_neurons"]),
        "tau": float(args["tau"]),
        "logic_layer_sizes": logic_layer_sizes,
        "gate_operations": list(ALL_OPERATIONS),
        "group_sum": {"type": "GroupSum", "class_count": int(class_count), "tau": float(args["tau"])},
        "preprocess": {
            "to_tensor": True,
            "binarization": binarization,
            "flatten_order": "row_major",
        },
        "csv_fields": ["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        "discretized_model": {
            "gates_csv": "best_lgn_gates.csv",
            "metadata_json": "best_lgn_metadata.json",
            "rule": "gate_id is argmax over the learned 16 gate logits for each neuron",
        },
    }
    if extra_metadata:
        meta.update(extra_metadata)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

def export_binarized_test_csv(out_path):
    all_x = []
    all_y = []
    for x, y in test_loader:
        xb = x.round().bool().reshape(x.shape[0], -1).to(torch.int8).cpu()
        all_x.append(xb)
        all_y.append(y.cpu())
    x_all = torch.cat(all_x, dim=0)
    y_all = torch.cat(all_y, dim=0)
    df = pd.DataFrame(x_all.numpy())
    df.insert(0, "label", y_all.numpy())
    df.to_csv(out_path, index=False)
    return df.shape

def write_json(path, value):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2)

def train_and_export_model(depth, width):
    args = make_args(depth, width)
    model_dir = SWEEP_OUTPUT_DIR / f"depth_{depth:02d}_width_{width:05d}"
    model_dir.mkdir(parents=True, exist_ok=True)

    done_files = [
        model_dir / "best_lgn_gates.csv",
        model_dir / "best_lgn_metadata.json",
        model_dir / "test_binarized.csv",
        model_dir / "training_summary.json",
    ]
    if SKIP_EXISTING and all(p.exists() for p in done_files):
        print(f"Skipping existing export: depth={depth}, width={width}, dir={model_dir}")
        with open(model_dir / "training_summary.json", "r", encoding="utf-8") as f:
            return json.load(f)

    set_seed(args["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, in_dim, total_num_neurons, total_num_weights = build_lgn(args, device)
    loss_fn = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args["learning_rate"])

    with open(model_dir / "log_args.txt", "w", encoding="utf-8") as f:
        f.write(json.dumps(args))

    print("=" * 80)
    print(f"Training depth={depth}, width={width}, tau={args['tau']}, device={device}")
    print("Model directory:", model_dir)
    print("in_dim:", in_dim, "total_num_neurons:", total_num_neurons, "total_num_weights:", total_num_weights)
    print("Connection plan:", json.dumps(getattr(model, "connection_plan", []), indent=2))

    best_val = -1.0
    best_epoch = 0
    best_model = deepcopy(model)
    wait = 0
    train_list = []
    val_list = []
    loss_list = []
    seconds_list = []
    cumulative_seconds = 0.0

    for epoch in range(args["epochs"]):
        start_time = time.time()
        model.train()
        last_loss = None

        for x, y in tqdm(train_loader, desc=f"depth={depth} width={width} epoch={epoch}"):
            x = x.to(torch.float32).to(device)
            y = y.to(device)
            last_loss = train_step(model, x, y, loss_fn, optimizer)

        cumulative_seconds += time.time() - start_time
        seconds_list.append(round(cumulative_seconds, 0))

        val_acc = eval_model(model, val_loader, mode=False, device=device)
        train_acc = eval_model(model, train_loader, mode=False, device=device)
        val_list.append(val_acc)
        train_list.append(train_acc)
        loss_list.append(last_loss)

        print(
            f"epoch={epoch:03d} loss={last_loss:.6f} "
            f"train_acc={train_acc:.6f} val_acc={val_acc:.6f} best_val={best_val:.6f}"
        )

        if val_acc > best_val:
            best_val = val_acc
            best_epoch = epoch
            best_model = deepcopy(model)
            wait = 0
        else:
            wait += 1

        if wait > args["patience"]:
            print(f"Early stopping at epoch {epoch}; best epoch was {best_epoch}.")
            break

    best_model = best_model.to(device)
    best_model.implementation = "cuda" if device == "cuda" else "python"
    test_acc = eval_model(best_model, test_loader, mode=False, device=device)

    model_name = f"Model_depth-{depth:02d}_width-{width:05d}_epoch-{best_epoch}"
    model_path = model_dir / model_name
    with open(model_path, "wb") as f:
        pickle.dump(best_model, f)

    write_json(model_dir / "log_seconds.txt", seconds_list)
    write_json(model_dir / "log_val.txt", val_list)
    write_json(model_dir / "log_train.txt", train_list)
    write_json(model_dir / "log_loss.txt", loss_list)

    export_lgn_gates_csv_and_json(
        best_model,
        model_dir / "best_lgn_gates.csv",
        model_dir / "best_lgn_metadata.json",
        args,
        CLASS_COUNT,
        extra_metadata={
            "sweep": {"depth": int(depth), "width": int(width), "run_id": SWEEP_RUN_ID},
            "best_epoch": int(best_epoch),
            "best_val_accuracy": float(best_val),
            "test_accuracy_python_discrete": float(test_acc),
            "model_file": model_name,
            "connection_plan": getattr(best_model, "connection_plan", []),
        },
    )
    test_csv_shape = export_binarized_test_csv(model_dir / "test_binarized.csv")

    result_text = (
        f"depth={depth}\n"
        f"width={width}\n"
        f"best_epoch={best_epoch}\n"
        f"best_val_accuracy={best_val:.16f}\n"
        f"test_accuracy_python_discrete={test_acc:.16f}\n"
        f"model_file={model_name}\n"
    )
    result_file = model_dir / f"results_boolean_depth-{depth:02d}_width-{width:05d}.txt"
    result_file.write_text(result_text, encoding="utf-8")

    summary = {
        "depth": int(depth),
        "width": int(width),
        "tau": float(args["tau"]),
        "best_epoch": int(best_epoch),
        "best_val_accuracy": float(best_val),
        "test_accuracy_python_discrete": float(test_acc),
        "training_seconds": float(cumulative_seconds),
        "model_dir": str(model_dir),
        "model_file": model_name,
        "test_csv_shape": list(test_csv_shape),
        "total_num_neurons": int(total_num_neurons),
        "total_num_weights": int(total_num_weights),
        "connection_plan": getattr(best_model, "connection_plan", []),
    }
    write_json(model_dir / "training_summary.json", summary)
    print(f"Saved model and discretized export to {model_dir}")
    print(f"Python discrete test accuracy: {test_acc:.6f}")

    del model, best_model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


In [ ]:
def make_args(depth, width):
    args = dict(BASE_ARGS)
    args.update({
        "num_layers": int(depth),
        "num_neurons": int(width),
        "tau": float(TAU_BY_WIDTH[int(width)]),
        "sweep_depth": int(depth),
        "sweep_width": int(width),
        "in_dim": int(INPUT_DIM),
        "class_count": int(CLASS_COUNT),
    })
    return args


def compute_in_dim(args):
    return int(args.get("in_dim", INPUT_DIM))


def export_lgn_gates_csv_and_json(model, csv_path, json_path, args, class_count, extra_metadata=None):
    rows = []
    for layer_idx, layer in enumerate(model):
        if isinstance(layer, LogicLayer):
            gate_ids = layer.weights.argmax(1).detach().cpu().tolist()
            a_idx = layer.indices[0].detach().cpu().tolist()
            b_idx = layer.indices[1].detach().cpu().tolist()
            for neuron_idx, (a, b, gate_id) in enumerate(zip(a_idx, b_idx, gate_ids)):
                rows.append({
                    "layer": int(layer_idx),
                    "neuron": int(neuron_idx),
                    "input_a": int(a),
                    "input_b": int(b),
                    "gate_id": int(gate_id),
                    "gate_name": str(ALL_OPERATIONS[gate_id]),
                })

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        )
        writer.writeheader()
        writer.writerows(rows)

    logic_layer_sizes = [int(layer.out_dim) for layer in model if isinstance(layer, LogicLayer)]
    meta = {
        "dataset": "UCI Phishing Websites",
        "dataset_source": DATA_URL,
        "uci_dataset_id": 327,
        "class_count": int(class_count),
        "class_names": [INDEX_TO_CLASS_NAME[i] for i in range(class_count)],
        "original_label_to_index": {str(k): int(v) for k, v in ORIGINAL_LABEL_TO_INDEX.items()},
        "target_column": TARGET_COLUMN,
        "feature_names": feature_names,
        "binary_feature_names": binary_feature_names,
        "input_encoding": {
            "type": "ternary_one_hot",
            "original_values": BASE_ARGS["original_feature_values"],
            "rule": "For each original feature, append indicators for value == -1, value == 0, value == 1 in that order.",
        },
        "in_dim": int(args["in_dim"]),
        "num_layers": int(args["num_layers"]),
        "num_neurons": int(args["num_neurons"]),
        "tau": float(args["tau"]),
        "logic_layer_sizes": logic_layer_sizes,
        "logic_layer_connection_modes": [str(getattr(layer, "connections", "unknown")) for layer in model if isinstance(layer, LogicLayer)],
        "connection_policy": {
            "requested_connections": args.get("requested_connections", "unique"),
            "fallback_connections": args.get("fallback_connections", "random"),
            "layer_plan": getattr(model, "connection_plan", []),
            "note": "Unique pairs are used when possible. For UCI Phishing, the first layer has only 90 binary inputs, so widths above 4005 require random pair reuse.",
        },
        "gate_operations": list(ALL_OPERATIONS),
        "group_sum": {"type": "GroupSum", "class_count": int(class_count), "tau": float(args["tau"])},
        "preprocess": {
            "missing_values": "none expected",
            "feature_encoding": "ternary one-hot",
            "flatten_order": "feature major, values -1/0/1 per feature",
        },
        "csv_fields": ["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        "discretized_model": {
            "gates_csv": "best_lgn_gates.csv",
            "metadata_json": "best_lgn_metadata.json",
            "rule": "gate_id is argmax over the learned 16 gate logits for each neuron",
        },
    }
    if extra_metadata:
        meta.update(extra_metadata)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)


def export_binarized_test_csv(out_path):
    all_x = []
    all_y = []
    for x, y_batch in test_loader:
        xb = x.round().bool().reshape(x.shape[0], -1).to(torch.int8).cpu()
        all_x.append(xb)
        all_y.append(y_batch.cpu())
    x_all = torch.cat(all_x, dim=0)
    y_all = torch.cat(all_y, dim=0)
    df = pd.DataFrame(x_all.numpy(), columns=binary_feature_names)
    df.insert(0, "label", y_all.numpy())
    df.to_csv(out_path, index=False)
    return df.shape


## Run the depth vs width training sweep

This is the long-running cell. It trains `6 * 4 = 24` models. With the default `epochs=200` and early stopping, the tabular dataset should be much faster than MNIST, but the widest/deepest models can still take meaningful GPU time.

For a smoke test, temporarily set `DEPTHS = [1]`, `WIDTHS = [2000]`, and `BASE_ARGS["epochs"] = 5` in the configuration cell, then restart the kernel and run the notebook again.

Connection note: the UCI feature vector has 90 binary inputs, so the first layer can have at most `90 choose 2 = 4005` unique input pairs. The notebook uses `unique` connections for widths 2000 and 4000, and automatically uses `random` only for first-layer widths 6000 and 8000 so the full table can still run. This is recorded in each model metadata file.


In [ ]:
sweep_summaries = []

if RUN_SWEEP:
    for depth in DEPTHS:
        for width in WIDTHS:
            summary = train_and_export_model(depth, width)
            sweep_summaries.append(summary)
            pd.DataFrame(sweep_summaries).to_csv(SWEEP_OUTPUT_DIR / "training_summary_so_far.csv", index=False)
            write_json(SWEEP_OUTPUT_DIR / "training_summary_so_far.json", sweep_summaries)
else:
    print("RUN_SWEEP is False; training was skipped.")

if sweep_summaries:
    summary_df = pd.DataFrame(sweep_summaries).sort_values(["depth", "width"])
    summary_df.to_csv(SWEEP_OUTPUT_DIR / "training_summary.csv", index=False)
    print(summary_df[["depth", "width", "best_epoch", "best_val_accuracy", "test_accuracy_python_discrete", "training_seconds", "model_dir"]].to_string(index=False))


## Validate exported artifacts

Run this after training to verify that every model folder contains the same artifact categories as the existing trained model folders.


In [ ]:
REQUIRED_ARTIFACTS = [
    "log_args.txt",
    "log_val.txt",
    "log_train.txt",
    "log_seconds.txt",
    "best_lgn_gates.csv",
    "best_lgn_metadata.json",
    "test_binarized.csv",
    "training_summary.json",
]

def find_exported_model_dirs(root):
    root = Path(root)
    return sorted({p.parent for p in root.rglob("best_lgn_metadata.json")})

exported_dirs = find_exported_model_dirs(SWEEP_OUTPUT_DIR)
validation_rows = []
for model_dir in exported_dirs:
    with open(model_dir / "best_lgn_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    missing = [name for name in REQUIRED_ARTIFACTS if not (model_dir / name).exists()]
    model_files = sorted(p.name for p in model_dir.glob("Model_depth-*_width-*_epoch-*"))
    validation_rows.append({
        "depth": meta["num_layers"],
        "width": meta["num_neurons"],
        "model_dir": str(model_dir),
        "model_files": ", ".join(model_files),
        "missing": ", ".join(missing),
    })

validation_df = pd.DataFrame(validation_rows)
if validation_df.empty:
    print(validation_df.to_string(index=False))
    raise RuntimeError("No exported model folders found. Run the training sweep first.")

validation_df = validation_df.sort_values(["depth", "width"])
print(validation_df.to_string(index=False))

if validation_df["missing"].str.len().gt(0).any():
    raise RuntimeError("Some model folders are missing required artifacts. See the validation table above.")
print("All exported model folders contain the required artifacts.")


## Rust plaintext evaluation for all discretized models

This cell runs the repository Rust evaluator on each exported `best_lgn_gates.csv` + `best_lgn_metadata.json` + `test_binarized.csv` folder. It evaluates the saved discretized models, prints a depth-by-width accuracy table, and saves report files into the sweep output directory.

Outputs:

- `rust_accuracy_report.csv`
- `rust_accuracy_report.json`
- `rust_accuracy_report.md`
- `rust_accuracy_pivot.csv`
- one `rust_eval_stdout.txt` per model folder


In [ ]:
import re

def latest_sweep_output_dir():
    candidates = [p for p in SWEEP_ROOT.iterdir() if p.is_dir()]
    if not candidates:
        raise RuntimeError("No sweep output directories found.")
    return max(candidates, key=lambda p: p.stat().st_mtime)

if "SWEEP_OUTPUT_DIR" in globals():
    EVAL_ROOT = Path(SWEEP_OUTPUT_DIR)
else:
    EVAL_ROOT = latest_sweep_output_dir()
print("Evaluating exported models under:", EVAL_ROOT)

def clean_cargo_env():
    """Avoid Conda MinGW compiler overrides when building the MSVC Rust target."""
    env = os.environ.copy()
    for key in [
        "CC", "CXX", "AR", "CFLAGS", "CXXFLAGS", "ARFLAGS",
        "HOST_CC", "HOST_CXX", "HOST_AR",
        "CC_x86_64_pc_windows_msvc", "CXX_x86_64_pc_windows_msvc", "AR_x86_64_pc_windows_msvc",
        "CC_x86_64-pc-windows-msvc", "CXX_x86_64-pc-windows-msvc", "AR_x86_64-pc-windows-msvc",
    ]:
        env.pop(key, None)
    return env

def lgn_eval_executable_path():
    exe_name = "lgn_eval.exe" if os.name == "nt" else "lgn_eval"
    return PROJECT_DIR / "crates" / "ei-ddlgn-eval" / "target" / "release" / exe_name

def build_rust_plaintext_evaluator():
    exe_path = lgn_eval_executable_path()
    if exe_path.exists():
        print("Using existing Rust evaluator:", exe_path)
        return exe_path
    cmd = ["cargo", "build", "--release", "--manifest-path", str(RUST_MANIFEST), "--bin", "lgn_eval"]
    print("Building Rust evaluator:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, env=clean_cargo_env(), capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError("Cargo build failed. See the printed Cargo stdout/stderr above.")
    print(proc.stdout)
    print(proc.stderr)
    if not exe_path.exists():
        raise RuntimeError(f"Cargo build finished but evaluator was not found: {exe_path}")
    return exe_path

RUST_EVAL_EXE = build_rust_plaintext_evaluator()

def run_rust_plaintext_eval(model_dir):
    cmd = [str(RUST_EVAL_EXE), str(model_dir)]
    print("Running:", " ".join(cmd))
    proc = subprocess.run(cmd, cwd=PROJECT_DIR, capture_output=True, text=True)
    output = proc.stdout + "\n" + proc.stderr
    (Path(model_dir) / "rust_eval_stdout.txt").write_text(output, encoding="utf-8")
    if proc.returncode != 0:
        raise RuntimeError(f"Rust evaluator failed for {model_dir}. See rust_eval_stdout.txt in that folder.")
    match = re.search(r"Accuracy:\s*([0-9]*\.?[0-9]+)", output)
    if not match:
        raise RuntimeError(f"Could not parse Accuracy from Rust output for {model_dir}.")
    return float(match.group(1)), output

model_dirs = find_exported_model_dirs(EVAL_ROOT)
if not model_dirs:
    raise RuntimeError("No exported model directories found for Rust evaluation.")

rust_rows = []
for model_dir in model_dirs:
    with open(model_dir / "best_lgn_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    accuracy, _ = run_rust_plaintext_eval(model_dir)
    rust_rows.append({
        "depth": int(meta["num_layers"]),
        "width": int(meta["num_neurons"]),
        "accuracy": accuracy,
        "accuracy_percent": accuracy * 100.0,
        "best_epoch": int(meta.get("best_epoch", -1)),
        "best_val_accuracy": float(meta.get("best_val_accuracy", float("nan"))),
        "python_discrete_accuracy": float(meta.get("test_accuracy_python_discrete", float("nan"))),
        "model_dir": str(model_dir),
    })

rust_df = pd.DataFrame(rust_rows).sort_values(["depth", "width"])
pivot = rust_df.pivot(index="depth", columns="width", values="accuracy_percent").sort_index().sort_index(axis=1)

report_csv = EVAL_ROOT / "rust_accuracy_report.csv"
report_json = EVAL_ROOT / "rust_accuracy_report.json"
report_md = EVAL_ROOT / "rust_accuracy_report.md"
pivot_csv = EVAL_ROOT / "rust_accuracy_pivot.csv"

rust_df.to_csv(report_csv, index=False)
rust_df.to_json(report_json, orient="records", indent=2)
pivot.to_csv(pivot_csv)

def markdown_table_from_df(df, float_format=".4f"):
    headers = [str(column) for column in df.columns]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for _, row in df.iterrows():
        values = []
        for value in row.tolist():
            if isinstance(value, float):
                values.append(format(value, float_format))
            else:
                values.append(str(value))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

report_lines = [
    "# UCI Phishing Websites DDLGN Depth vs Width Rust Accuracy Report",
    "",
    f"Generated: {datetime.datetime.now().isoformat(timespec='seconds')}",
    f"Evaluation root: `{EVAL_ROOT}`",
    "",
    "## Per-model results",
    "",
    markdown_table_from_df(rust_df[["depth", "width", "accuracy", "accuracy_percent", "best_epoch", "model_dir"]]),
    "",
    "## Accuracy percent pivot",
    "",
    markdown_table_from_df(pivot.reset_index().round(4)),
    "",
]
report_md.write_text("\n".join(report_lines), encoding="utf-8")

print("Rust accuracy percent table:")
print(pivot.round(2).to_string())
print("Saved:", report_csv)
print("Saved:", report_json)
print("Saved:", report_md)
print("Saved:", pivot_csv)
